# Hibiki-Sw Stage 1: Text Adaptation

**GPU Budget: ~4 hours** (15k steps, 2x T4)

Continue-pretrain the Temporal Transformer on en+sw text using
next-token prediction. This builds the bilingual language model
backbone before introducing audio.

**Prerequisites:**
- Notebook 00 completed (tokenizer + text tokens)
- `hibiki-sw-data` Kaggle dataset attached

In [ ]:
!pip install -q transformers accelerate sentencepiece pyyaml tensorboard

In [ ]:
import os
import torch

# Verify 2x T4 GPUs
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} "
          f"({torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB)")

REPO_DIR = "/kaggle/input/datasets/victormugambi/hibiki-sw/hibiki-sw"
DATA_DIR = "/kaggle/input/hibiki-sw-data"
OUTPUT_DIR = "/kaggle/working/stage1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Copy code to working dir (needed for torchrun)
!cp -r {REPO_DIR}/* /kaggle/working/hibiki-sw/
!ls /kaggle/working/hibiki-sw/

In [ ]:
%%time
# Launch Stage 1 training with DDP on 2 GPUs
!cd /kaggle/working/hibiki-sw && torchrun \
    --nproc_per_node=2 \
    --master_port=29500 \
    training/train_text.py \
    --config configs/model_100m.yaml \
    --data_dir {DATA_DIR}/text_tokens \
    --output_dir {OUTPUT_DIR}

In [ ]:
# Verify checkpoint
ckpt = torch.load(f"{OUTPUT_DIR}/checkpoint_final.pt", map_location="cpu")
print(f"Final step: {ckpt['step']}")
print(f"Model keys: {len(ckpt['model'])}")
total_params = sum(v.numel() for v in ckpt['model'].values())
print(f"Total parameters: {total_params:,}")

In [ ]:
# Optional: upload checkpoint to HuggingFace Hub for persistence
# from huggingface_hub import HfApi
# api = HfApi(token="YOUR_TOKEN")
# api.create_repo("YOUR_USER/hibiki-sw-stage1", exist_ok=True, private=True)
# api.upload_file(
#     path_or_fileobj=f"{OUTPUT_DIR}/checkpoint_final.pt",
#     path_in_repo="checkpoint_final.pt",
#     repo_id="YOUR_USER/hibiki-sw-stage1",
# )
print("Stage 1 complete!")